# Set-up

In [1]:
%%capture
!pip install torch transformers
!pip install -U "huggingface_hub[cli]"

# (recommended inside a fresh venv/conda env)
!pip uninstall -y huggingface_hub
!pip install "huggingface-hub>=0.34.0,<1.0"
# sanity check
!python -c "import transformers, huggingface_hub as h; print('transformers', transformers.__version__, 'hub', h.__version__)"

In [2]:
import os
os.environ["HF_HOME"] = "./hf"
os.environ["HF_HUB_CACHE"] = "./hf/hub"
os.environ['PATH'] += ':/storage/home/hcoda1/9/kzhang430/.local/bin'
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
import re
import json
from huggingface_hub import login

In [1]:
pwd

'/storage/scratch1/9/kzhang430'

# Generating synthetic datasets

## Factors:
1. Magnitude of operands
2. Number of operands
3. Distance between operands
4. Number of carries needed

In [ ]:
cd ./data

In [18]:
os.mkdir('magnitude')
os.mkdir('number')
os.mkdir('distance')
os.mkdir('carries')

## Magnitude

In [ ]:
cd ./magnitude

## Finding distribution of math data (from the Pile)

In [ ]:
login()

In [3]:
!hf auth whoami

user:  wristycargo


In [4]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable


In [5]:
from datasets import load_dataset_builder
ds_builder = load_dataset_builder("monology/pile-uncopyrighted-parquet")

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

In [ ]:
ds_builder.info.features

In [ ]:
import re
from datasets import load_dataset

# ----------------------------
# 1) Helpers: line window printer
# ----------------------------
def get_line_window(text: str, start: int, end: int, radius: int = 2):
    """
    Return a small block of lines around the match span.
    Lines are 1-indexed in the returned l1/l2.
    """
    lines = text.splitlines()
    if not lines:
        return "", 0, 0

    # Map character index -> line index
    char = 0
    start_line = 0
    end_line = 0
    for idx, line in enumerate(lines):
        line_len = len(line) + 1  # + newline
        if char <= start < char + line_len:
            start_line = idx
        if char <= end <= char + line_len:
            end_line = idx
            break
        char += line_len
    else:
        # If end is past last newline, clamp
        end_line = len(lines) - 1

    lo = max(0, start_line - radius)
    hi = min(len(lines) - 1, end_line + radius)

    block = "\n".join(f"{j+1:>4}| {lines[j]}" for j in range(lo, hi + 1))
    return block, lo + 1, hi + 1


# ----------------------------
# 2) Extract chain equalities and verify (safe parser, no eval)
# ----------------------------

CHAIN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9_.])                     # safe-ish left boundary (avoid identifiers, decimals)
    (?P<chain>
        (?P<lhs>                           # expression side: must contain a digit
            (?=[\d\s,()+\-−*/×÷]*\d)       # lookahead: at least one digit somewhere
            [\d\s,()+\-−*/×÷]+
        )
        \s*=\s*
        (?P<rhs>
            (?=[\d\s,()+\-−*/×÷]*\d)
            [\d\s,()+\-−*/×÷]+
        )
        (?:                                # optional extra = expr parts
            \s*=\s*
            (?:
                (?=[\d\s,()+\-−*/×÷]*\d)
                [\d\s,()+\-−*/×÷]+
            )
        )*
    )
    (?![A-Za-z0-9_.])                      # safe-ish right boundary
    """,
    re.VERBOSE
)

def _norm_ops(s: str) -> str:
    return (s.replace("−", "-")
             .replace("×", "*")
             .replace("÷", "/"))

def _tokenize(expr: str):
    expr = _norm_ops(expr).replace(",", "")
    tokens = []
    i = 0
    prev = None  # None, 'num', 'op', '(' , ')'
    while i < len(expr):
        ch = expr[i]
        if ch.isspace():
            i += 1
            continue

        if ch.isdigit():
            j = i
            while j < len(expr) and expr[j].isdigit():
                j += 1
            tokens.append(("num", int(expr[i:j])))
            prev = "num"
            i = j
            continue

        if ch in "+-*/()":
            # unary minus support: "-5", "3*-2", "(-7)"
            if ch == "-" and (prev is None or prev in ("op", "(")):
                j = i + 1
                while j < len(expr) and expr[j].isspace():
                    j += 1
                if j < len(expr) and expr[j].isdigit():
                    k = j
                    while k < len(expr) and expr[k].isdigit():
                        k += 1
                    tokens.append(("num", -int(expr[j:k])))
                    prev = "num"
                    i = k
                    continue
                # treat as 0 - ...
                tokens.append(("num", 0))
                tokens.append(("op", "-"))
                prev = "op"
                i += 1
                continue

            if ch == "(":
                tokens.append(("lp", ch)); prev = "("
            elif ch == ")":
                tokens.append(("rp", ch)); prev = ")"
            else:
                tokens.append(("op", ch)); prev = "op"
            i += 1
            continue

        # unexpected char (letters/units/etc) -> reject this expr
        return None

    return tokens

def _to_rpn(tokens):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    out = []
    stack = []
    for ttype, val in tokens:
        if ttype == "num":
            out.append((ttype, val))
        elif ttype == "op":
            while stack and stack[-1][0] == "op" and prec[stack[-1][1]] >= prec[val]:
                out.append(stack.pop())
            stack.append((ttype, val))
        elif ttype == "lp":
            stack.append((ttype, val))
        elif ttype == "rp":
            while stack and stack[-1][0] != "lp":
                out.append(stack.pop())
            if not stack or stack[-1][0] != "lp":
                return None
            stack.pop()
        else:
            return None

    while stack:
        if stack[-1][0] in ("lp", "rp"):
            return None
        out.append(stack.pop())

    return out

def _eval_rpn(rpn):
    st = []
    for ttype, val in rpn:
        if ttype == "num":
            st.append(val)
            continue

        if len(st) < 2:
            return None
        b = st.pop()
        a = st.pop()

        if val == "+":
            st.append(a + b)
        elif val == "-":
            st.append(a - b)
        elif val == "*":
            st.append(a * b)
        elif val == "/":
            # only accept exact integer division
            if b == 0 or a % b != 0:
                return None
            st.append(a // b)
        else:
            return None

    return st[0] if len(st) == 1 else None

def eval_int_expr(expr: str):
    toks = _tokenize(expr)
    if toks is None:
        return None
    rpn = _to_rpn(toks)
    if rpn is None:
        return None
    return _eval_rpn(rpn)

def extract_verified_chains(text: str, output_unverified: bool = False):
    matches = []
    v_cnt = 0
    u_cnt = 0

    for m in CHAIN_RE.finditer(text):
        chain = m.group("chain").strip()

        # reject empty-ish junk
        if "=" not in chain:
            continue

        # reject assignment operators and common non-math equality patterns
        if "+=" in chain or "-=" in chain or "*=" in chain or "/=" in chain:
            continue
        if "==" in chain or "=>" in chain or "=<" in chain:
            continue
        parts = [p.strip() for p in chain.split("=")]
        vals = [eval_int_expr(p) for p in parts]
        verified = (None not in vals) and all(v == vals[0] for v in vals[1:])

        if verified:
            v_cnt += 1
        else:
            u_cnt += 1

        if verified or output_unverified:
            matches.append({
                "expr": chain,
                "verified": verified,
                "parts": parts,
                "values": vals,
                "span": (m.start("chain"), m.end("chain")),
            })

    return matches, v_cnt, u_cnt


# ----------------------------
# 3) Your streaming loop (unchanged)
# ----------------------------
ds = load_dataset("monology/pile-uncopyrighted-parquet", split="train", streaming=True)

max_rows = 1000
total_verified = 0
total_unverified = 0

for i, ex in enumerate(ds):
    text = ex["text"]
    matches, v_cnt, u_cnt = extract_verified_chains(text, output_unverified=True)

    if matches:
        print(f"\n========== ROW {i} ({len(matches)} matches) ==========")
        for k, mm in enumerate(matches, 1):
            s, e = mm["span"]
            block, l1, l2 = get_line_window(text, s, e, radius=2)
            print(f"\n--- match {k}: {mm['expr']} | verified={mm['verified']} | lines {l1}-{l2} ---")
            if mm["verified"]:
                print(f"    parts={mm['parts']}  values={mm['values']}")
            else:
                print(f"    parts={mm['parts']}  values={mm['values']}")
            print(block)

    total_verified += v_cnt
    total_unverified += u_cnt

    if i >= max_rows - 1:
        break

print("\ndone ->")
print(f"Total verified expressions: {total_verified}")
print(f"Total unverified expressions: {total_unverified}")


In [5]:
ds = load_dataset("monology/pile-uncopyrighted-parquet", split="train", streaming=True)

max_rows = 100
total_verified = 0
total_unverified = 0

for i, ex in enumerate(ds):
    text = ex["text"]
    matches, v_cnt, u_cnt = extract_verified_chains(text, output_unverified=True)

    if matches:
        print(f"\n========== ROW {i} ({len(matches)} matches) ==========")
        for k, mm in enumerate(matches, 1):
            s, e = mm["span"]
            block, l1, l2 = get_line_window(text, s, e, radius=2)
            print(f"\n--- match {k}: {mm['expr']} | verified={mm['verified']} | lines {l1}-{l2} ---")
            print(block)

    total_verified += v_cnt
    total_unverified += u_cnt

    if i >= max_rows - 1:
        break

print("\ndone ->")
print(f"Total verified expressions: {total_verified}")
print(f"Total unverified expressions: {total_unverified}")


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]


========== ROW 42 (4 matches) ==========


NameError: name 'get_line_window' is not defined

In [96]:
ds = load_dataset("monology/pile-uncopyrighted-parquet", split="train", streaming=True)

out_path = "pile_arith_matches.jsonl"
max_rows = 100

total_verified = 0
total_unverified = 0

for i, ex in enumerate(ds):
    text = ex["text"]
    matches, verified_count, unverified_count = extract_verified_expressions(text, output_unverified = True)
    if matches:
        for mm in matches:
            print(mm)
    
    total_verified += verified_count
    total_unverified += unverified_count
    if i >= max_rows - 1:
        break

# with open(out_path, "w", encoding="utf-8") as f:
#     for i, ex in enumerate(ds):
#         text = ex["text"]
#         matches = extract_verified_expressions(text)
#         if matches:
#             # store the full row text OR just the local context; your choice
#             for mm in matches:
#                 s, e = mm["span"]
#                 context = text[max(0, s-80):min(len(text), e+80)]
#                 rec = {
#                     "row_index": i,
#                     "expr": mm["expr"],
#                     "a": mm["a"], "op": mm["op"], "b": mm["b"], "c": mm["c"],
#                     "context": context,
#                 }
#                 f.write(json.dumps(rec, ensure_ascii=False) + "\n")

#         if i >= max_rows - 1:
#             break

print("done ->")
print(f"Total verified expressions: {total_verified}")
print(f"Total unverified expressions: {total_unverified}")

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

{'expr': '50 / 2 = 840', 'a': 50, 'b': 2, 'c': 840, 'op': '/'}
{'expr': '50 / 2 = 840', 'a': 50, 'b': 2, 'c': 840, 'op': '/'}
done ->
Total verified expressions: 0
Total unverified expressions: 2


In [ ]:
for i, ex in enumerate(ds):
    text = ex["text"]
    matches = extract_un_verified_expressions(text)
    if matches:
        for mm in matches:
            print(mm)
    if i >= max_rows - 1:
        break

In [53]:
from datasets import get_dataset_split_names
get_dataset_split_names("monology/pile-uncopyrighted-parquet")

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

['train']

# Used Code

In [3]:
CAND_RE = re.compile(
    r"""
    (?<![\w.])                       # don't start mid-word / after a dot in decimals
    (?P<a>[+\-−]?\d{1,3}(?:,\d{3})*|\d+)   # A
    \s*(?P<op>[+\-*/×÷])\s*                # operator
    (?P<b>[+\-−]?\d{1,3}(?:,\d{3})*|\d+)   # B
    \s*=\s*
    (?P<c>[+\-−]?\d{1,3}(?:,\d{3})*|\d+)   # C
    (?![\w.])                        # don't end mid-word / before a dot in decimals
    """,
    re.VERBOSE,
)

def _to_int(s: str) -> int:
    s = s.replace("−", "-").replace(",", "")
    return int(s)

def _apply(a: int, b: int, op: str):
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op in ("*", "×"):
        return a * b
    if op in ("/", "÷"):
        # Only keep "clean" integer divisions to avoid float noise
        if b == 0:
            return None
        if a % b != 0:
            return None
        return a // b
    return None


def extract_verified_expressions(text: str):
    out = []
    verified_count = 0
    unverified_count = 0

    for m in CAND_RE.finditer(text):
        a_s, b_s, c_s, op = m.group("a"), m.group("b"), m.group("c"), m.group("op")
        try:
            a, b, c = _to_int(a_s), _to_int(b_s), _to_int(c_s)
        except ValueError:
            continue

        val = _apply(a, b, op)
        if val is None or val != c:
            unverified_count += 1
            continue

        out.append({
            "expr": f"{a_s} {op} {b_s} = {c_s}",
            "a": a, "b": b, "c": c, "op": op,
            })
        
        verified_count += 1
        unverified_count += 1
    return out, verified_count, unverified_count



error_count = 0

In [4]:
cd data

/storage/scratch1/9/kzhang430/data


In [ ]:
from datasets import load_dataset, Value
from tqdm import tqdm
import json

ds = load_dataset("monology/pile-uncopyrighted-parquet", split="train", streaming=False)

START = 100_000_000


total_verified = 0
total_unverified = 0
out_path = "pile_arith_matches_full_againhmmm.jsonl"

with open(out_path, "w", encoding="utf-8") as f:
    for i, ex in tqdm(enumerate(ds[START:]), total=len(ds) - START):
        row_index = START + i

        text = ex["text"]                    

        matches, verified_count, unverified_count = extract_verified_expressions(text)

        if matches:
            for mm in matches:
                rec = {
                    "row_index": row_index,
                    "expr": mm["expr"],
                    "a": mm["a"], "op": mm["op"], "b": mm["b"], "c": mm["c"],
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                print('match')

        total_verified += verified_count
        total_unverified += unverified_count

print("done ->")
print(f"Total verified expressions: {total_verified}")
print(f"Total unverified expressions: {total_unverified}")


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/1976 [00:00<?, ?it/s]

Issues:

- Pile is very big, even analyzing 10% takes hours, but max time on pace is 8 hours
- Have to ignore many expressions, or how to handle certain expression that appear in text:
1. algebraic expressions: -5*v - 120 = 3*v
2. Equations in science/statistical statements, e.g. 𝑡_1,9=3,57,𝑃<.01
3. 

- experiment with metrics for problem size and find the distribution in the corpus
    - e.g. product of a and b
    - sum of number of digits across all 3
    - maybe 2 smallest of a,b,c
    - think 
 - different operands
 - 

In [13]:
from datasets import load_dataset, Value
from tqdm import tqdm
import json

ds = load_dataset("monology/pile-uncopyrighted-parquet", split="train", streaming=False)

START = 34859562  # <-- 100k (0-indexed). Use 10_000_000 if you really meant ten million.

# Prevent Arrow from UTF-8 decoding on access
ds = ds.cast_column("text", Value("binary"))

total_verified = 0
total_unverified = 0
out_path = "pile_arith_matches_full_fresh.jsonl"



with open(out_path, "w", encoding="utf-8") as f:
    for i, ex in tqdm(enumerate(ds[START:]), total=len(ds) - START):
        row_index = START + i

        b = ex["text"]                      # bytes
        text = b.decode("utf-8", errors="replace")  # or errors="ignore"
        

        matches, verified_count, unverified_count = extract_verified_expressions(text)

        if matches:
            for mm in matches:
                rec = {
                    "row_index": row_index,
                    "expr": mm["expr"],
                    "a": mm["a"], "op": mm["op"], "b": mm["b"], "c": mm["c"],
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        total_verified += verified_count
        total_unverified += unverified_count

print("done ->")
print(f"Total verified expressions: {total_verified}")
print(f"Total unverified expressions: {total_unverified}")


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/1976 [00:00<?, ?it/s]

Casting the dataset:   0%|          | 0/177009652 [00:00<?, ? examples/s]

KeyboardInterrupt: 

# Trying out FineWeb 

https://huggingface.co/datasets/HuggingFaceFW/fineweb

Using streaming

In [6]:
CAND_RE = re.compile(
    r"""
    (?<![\w.])                       # don't start mid-word / after a dot in decimals
    (?P<a>[+\-−]?\d{1,3}(?:,\d{3})*|\d+)   # A
    \s*(?P<op>[+\-*/×÷])\s*                # operator
    (?P<b>[+\-−]?\d{1,3}(?:,\d{3})*|\d+)   # B
    \s*=\s*
    (?P<c>[+\-−]?\d{1,3}(?:,\d{3})*|\d+)   # C
    (?![\w.])                        # don't end mid-word / before a dot in decimals
    """,
    re.VERBOSE,
)

def _to_int(s: str) -> int:
    s = s.replace("−", "-").replace(",", "")
    return int(s)

def _apply(a: int, b: int, op: str):
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op in ("*", "×"):
        return a * b
    if op in ("/", "÷"):
        # Only keep "clean" integer divisions to avoid float noise
        if b == 0:
            return None
        if a % b != 0:
            return None
        return a // b
    return None


def extract_verified_expressions(text: str):
    out = []
    verified_count = 0
    unverified_count = 0

    for m in CAND_RE.finditer(text):
        a_s, b_s, c_s, op = m.group("a"), m.group("b"), m.group("c"), m.group("op")
        try:
            a, b, c = _to_int(a_s), _to_int(b_s), _to_int(c_s)
        except ValueError:
            continue

        val = _apply(a, b, op)
        if val is None or val != c:
            unverified_count += 1
            continue

        out.append({
            "expr": f"{a_s} {op} {b_s} = {c_s}",
            "a": a, "b": b, "c": c, "op": op,
            })
        
        verified_count += 1
        unverified_count += 1
    return out, verified_count, unverified_count



error_count = 0

In [8]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import json

START = 0
N = 1_000_0000

fw = load_dataset("HuggingFaceFW/fineweb", name="CC-MAIN-2024-10",
                  split="train", streaming=True)

fw = fw.skip(START).take(N)  # streaming-friendly split :contentReference[oaicite:5]{index=5}

dl = DataLoader(fw, num_workers=4, prefetch_factor=10, batch_size=None)

out_path = "pile_arith_matches_full_fresh.jsonl"
total_verified = 0
total_unverified = 0

with open(out_path, "w", encoding="utf-8") as f:
    for row_index, ex in tqdm(enumerate(dl, start=START), total=N):
        text = ex["text"]  # already str for fineweb streaming :contentReference[oaicite:6]{index=6}

        matches, verified_count, unverified_count = extract_verified_expressions(text)

        for mm in matches:
            rec = {"row_index": row_index, "expr": mm["expr"], "a": mm["a"], "op": mm["op"], "b": mm["b"], "c": mm["c"]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        total_verified += verified_count
        total_unverified += unverified_count

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/300 [00:00<?, ?it/s]

100%|██████████| 10000000/10000000 [43:22<00:00, 3842.10it/s] 


## Trying out dedupe

In [10]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import json

START = 0
N = 1_000_0000

fw = load_dataset("EleutherAI/the_pile_deduplicated", name="CC-MAIN-2024-10",
                  split="train", streaming=True)

fw = fw.skip(START).take(N)  # streaming-friendly split :contentReference[oaicite:5]{index=5}

dl = DataLoader(fw, num_workers=4, prefetch_factor=10, batch_size=None)

out_path = "pile_arith_matches_full_fresh_deduped.jsonl"
total_verified = 0
total_unverified = 0

with open(out_path, "w", encoding="utf-8") as f:
    for row_index, ex in tqdm(enumerate(dl, start=START), total=N):
        text = ex["text"]  # already str for fineweb streaming :contentReference[oaicite:6]{index=6}

        matches, verified_count, unverified_count = extract_verified_expressions(text)

        for mm in matches:
            rec = {"row_index": row_index, "expr": mm["expr"], "a": mm["a"], "op": mm["op"], "b": mm["b"], "c": mm["c"]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        total_verified += verified_count
        total_unverified += unverified_count

Resolving data files:   0%|          | 0/1650 [00:00<?, ?it/s]

ValueError: BuilderConfig 'CC-MAIN-2024-10' not found. Available: ['default']

In [9]:
from datasets import load_dataset, Value
from tqdm.auto import tqdm
import json

ds = load_dataset("EleutherAI/the_pile_deduplicated", split="train", streaming=True)

START = 0
ds = ds.cast_column("text", Value("binary"))

# Try to get a total for tqdm (may or may not be available in streaming)
total = None
try:
    total = ds.info.splits["train"].num_examples - START
except Exception:
    pass

total_verified = 0
total_unverified = 0
out_path = "pile_arith_matches_full_fresh_deduped.jsonl"

stream = ds.skip(START)

with open(out_path, "w", encoding="utf-8") as f:
    for row_index, ex in tqdm(enumerate(stream, start=START), total=total, desc="scan"):
        b = ex["text"]
        if isinstance(b, memoryview):  # sometimes binary comes back as memoryview
            b = b.tobytes()

        text = b.decode("utf-8", errors="replace")

        matches, verified_count, unverified_count = extract_verified_expressions(text)

        for mm in matches:
            rec = {
                "row_index": row_index,
                "expr": mm["expr"],
                "a": mm["a"], "op": mm["op"], "b": mm["b"], "c": mm["c"],
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        total_verified += verified_count
        total_unverified += unverified_count

print("done ->")
print(f"Total verified expressions: {total_verified}")
print(f"Total unverified expressions: {total_unverified}")

Resolving data files:   0%|          | 0/1650 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

TypeError: object of type 'IterableDataset' has no len()